# A Scanner for Simple Arithmetic Expressions

This notebook shows how a scanner can be implemented using nothing but Python's built-in `re` module &mdash;
no external scanner-generator library is needed.  We build a scanner that tokenizes simple arithmetic
expressions such as
```
3 + 4 * 10 + 007 + (-20) * 2
```
into a sequence of tokens like `NUMBER`, `PLUS`, `TIMES`, and so on.

The general recipe consists of three steps:
1. We decide on the names of all token types that our scanner is supposed to recognize.
2. For every token name, we specify a regular expression that describes the set of strings that are to be
   recognized as tokens of this type.
3. We combine all of these regular expressions into a single *master pattern* and use this pattern to scan
   the input string from left to right.

## Imports

We need the module `re` for regular expressions, and the module `collections`, which provides
`namedtuple`&mdash;a convenient way to define a lightweight class for our tokens.

In [ ]:
import re
import collections

## Specifying the Tokens

The *token specification* below is a list of pairs.  The first component of every pair is the name of a
token, written in capital letters by convention.  The second component is a regular expression that
describes the strings which are to be recognized as tokens of this type.

Three of the names occurring below do not name genuine tokens.  Instead, they serve a bookkeeping purpose:
- `NEWLINE` matches a single newline character.  Whenever this token is found, we have to increment our
  line counter.  The newline character itself is then discarded.
- `SKIP` matches blanks and tabs.  These characters only separate tokens and are therefore discarded.
- `MISMATCH` matches any single character that has not already been matched by one of the preceding
  regular expressions.  If this token is found, an unexpected character has occurred in the input and the
  scanner has to report an error.

**Note:** Python's `re` module tries the alternatives of a regular expression from left to right and picks
the *first* one that matches at the current position&mdash;unlike some scanner generators, which pick the
*longest* match irrespective of order.  Therefore, the order of the pairs below is significant: more
specific patterns have to be listed before more general ones.

In [ ]:
token_specification = [
    ('NUMBER',   r'0|[1-9][0-9]*'),
    ('PLUS',     r'\+'),
    ('MINUS',    r'-'),
    ('TIMES',    r'\*'),
    ('DIVIDE',   r'/'),
    ('LPAREN',   r'\('),
    ('RPAREN',   r'\)'),
    ('NEWLINE',  r'\n'),
    ('SKIP',     r'[ \t]+'),
    ('MISMATCH', r'.'),
]

## Building the Master Pattern

We combine the individual regular expressions into a single *master pattern*.  Every regular expression is
wrapped inside a *named group*, which is written as
```
(?P<NAME>regexp)
```
These named groups are then joined using the alternative operator `|`.  Later, once a match has been found,
we can ask which of the named groups is responsible for the match.

In [ ]:
master_pattern = '|'.join(f'(?P<{name}>{regexp})' for name, regexp in token_specification)

Let's take a look at the master pattern that has been generated:

In [ ]:
master_pattern

## Representing Tokens

We define the class `Token` using `collections.namedtuple`.  Every object of this class has four
attributes, namely `type`, `value`, `line`, and `column`.

In [ ]:
Token = collections.namedtuple('Token', ['type', 'value', 'line', 'column'])

## The Scanner

The function `tokenize` is a *generator function*: instead of returning a single value, it `yield`s a
sequence of `Token` objects, one for every token found in its argument `code`.

- `line_number` counts the lines of the input, starting at $1$.  `line_start` stores the position at which
  the current line begins; it is needed to compute the column of a token.
- `re.finditer` scans the string `code` for non-overlapping matches of `master_pattern`.  It returns an
  iterator of *match objects*, one for every token that is found, in the order in which these tokens occur
  in the input. A *match object* has the following attributes and methods:
  1. The attribute `mo.lastgroup` of the match object `mo` tells us the name of the named group responsible
     for the match, i.e. the name of the token that has been found.
  2. The method `mo.group()` extracts the string that has been matched.
  3. The method `mo.start()` returns the position at which the match begins.
  4. The method `mo.end()` returns the position at which the match begins.
- `kind` is the name of the token,
- `value` is the string that has been matched.
- `column` is computed by subtracting `line_start` from the position at which the match begins.
- The token `NUMBER` is handled by converting the matched string into an `int`.
- The token `NEWLINE` updates `line_start`, increments `line_number`, and then continues the loop with
  `continue`&mdash;no token is generated for a newline.
- The token `SKIP` is simply discarded by continuing the loop.
- The token `MISMATCH` raises an exception that reports the illegal character together with the line and
  column at which it was found.
- Every other match reaches the final line of the loop body, where a `Token` object is constructed and
  `yield`ed to the caller.

In [ ]:
def tokenize(code):
    line_number = 1
    line_start  = 0
    for mo in re.finditer(master_pattern, code):
        kind   = mo.lastgroup
        value  = mo.group()
        column = mo.start() - line_start
        if kind == 'NUMBER':
            value = int(value)
        elif kind == 'NEWLINE':
            line_start   = mo.end()
            line_number += 1
            continue
        elif kind == 'SKIP':
            continue
        elif kind == 'MISMATCH':
            raise RuntimeError(f"Illegal character {value!r} at line {line_number}, column {column}.")
        yield Token(kind, value, line_number, column)

## Trying it Out

We define a sample arithmetic expression that we want to tokenize.

In [ ]:
data = '''
3 + 4 * 10 + 007 +
(-20) * 2
'''
data

We call `tokenize(data)` and print every token that is found.  Since `tokenize` is a generator function,
calling it does not run any code yet; the code only runs while we iterate over the result, one token at a
time.

In [ ]:
for tok in tokenize(data):
    print(tok)

## Token Attributes

The scanner returns tokens as instances of the class `Token`, which possess four distinct attributes:
- The **`type`** attribute indicates the type of the token.  It holds a string value that corresponds to
  one of the names occurring in the token specification.
- The **`value`** attribute usually contains the recognized string.  However, as shown in the case of the
  token `NUMBER`, this value can be transformed before the token is constructed.
- The **`line`** attribute specifies the line number at which the token was found.
- The **`column`** attribute specifies the column at which the token starts.  Note that, unlike a counter
  that simply counts characters from the very beginning of the input, this column is reset to $0$ at the
  start of every line, which matches the way most editors report cursor positions.

## What Happens with an Unexpected Character?

Finally, let's see what happens if the input contains a character that is not covered by any of the
regular expressions in `token_specification`, for example the character `#`.

In [ ]:
try:
    for tok in tokenize('3 + 4 # 2'):
        print(tok)
except RuntimeError as error:
    print(error)